# Đánh giá ablation (feature 003) trên Colab GPU

Chạy **đúng code của branch `hocnv/DO-06`** + **model v1** (`Qwen3-1.7B.Q4_K_M.gguf`, cùng file với `merged_model/model.gguf` local) với `POSTPROCESS_ENABLED=false` (biến thể A), rồi mở Cloudflare Tunnel để máy local chạy `eval.predict`.

1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Runtime → **Run all**.
3. Copy URL `https://....trycloudflare.com` ở cell cuối gửi lại cho Claude. Giữ tab mở tới khi chạy xong.


In [ ]:
# [1] Code đúng branch
REPO = "https://github.com/long20102004/recruit_system.git"
BRANCH = "hocnv/DO-06"
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
%cd /content/recruit_system/ai-service
!git log --oneline -1


In [ ]:
# [2] Thư viện: llama-cpp-python bản CUDA 12 + CUDA 12 runtime từ pip (Colab có thể không còn libcudart.so.12)
!pip install -q fastapi uvicorn pydantic json-repair rapidfuzz gdown
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Đăng ký thư mục lib của các gói nvidia-* (pip) với dynamic linker để mọi process đều thấy libcudart/libcublas
import glob, site, os
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
!ldconfig
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA libs:", lib_dirs)
!ldconfig -p | grep -E "libcudart.so.12|libcublas.so.12" | head -3
# Kiểm tra trong process mới (tránh cache lỗi import của kernel hiện tại)
!python -c "import llama_cpp; print('llama_cpp', llama_cpp.__version__, 'GPU offload:', llama_cpp.llama_supports_gpu_offload())"


In [ ]:
# [3] Model v1 từ Drive dự án (public): finetune_output/gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf
import os
os.makedirs("merged_model", exist_ok=True)
if not os.path.exists("merged_model/model.gguf"):
    !gdown -q 1Kt4BbL22tR1HLw16f2tyGyus6mQDI2SV -O merged_model/model.gguf
print(os.path.getsize("merged_model/model.gguf"), "bytes (kỳ vọng 1107409216)")


In [ ]:
# [4] Chạy service (A: tắt hậu kiểm) + tunnel
import subprocess, time, re, urllib.request, json, os
env = dict(os.environ, POSTPROCESS_ENABLED="false", GGUF_MODEL_PATH="./merged_model/model.gguf")
srv = subprocess.Popen(["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(120):
    try:
        h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")); break
    except Exception:
        time.sleep(2)
print("health:", h)
assert h["engine"] == "llama_cpp", "GGUF chưa nạp được, xem server.log"

tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: url = m.group(0); break
print("=" * 60); print("URL gửi cho Claude:", url); print("=" * 60)


In [ ]:
# [5] Giữ session sống + theo dõi (dừng cell này khi xong)
import time
while True:
    n = sum(1 for l in open("server.log") if "Inference time" in l)
    print(time.strftime("%H:%M:%S"), "requests done:", n, flush=True)
    time.sleep(60)
